In [ ]:
import os
import sys
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import wandb
from tqdm import tqdm

# ==========================================================
# 1. HYBRID NOTEBOOK/SCRIPT PATH RESOLUTION ENGINE
# ==========================================================
try:
    # If running as a standard script, __file__ exists
    current_file_dir = os.path.dirname(os.path.abspath(__file__))
    BASE_DIR = os.path.abspath(os.path.join(current_file_dir, "..", ".."))
    print("--- Console: Running as a standard Python script ---")
except NameError:
    # running inside a Jupyter Notebook, handle path fallback cleanly
    print("--- Console: Running inside a Jupyter Notebook kernel ---")
    current_dir = os.getcwd()
    BASE_DIR = None
    # Traverse up to 5 levels to locate the repository root folder dynamically
    for _ in range(5):
        if os.path.exists(os.path.join(current_dir, 'src')) or os.path.exists(os.path.join(current_dir, 'notebooks')):
            BASE_DIR = current_dir
            break
        current_dir = os.path.dirname(current_dir)
    
    if BASE_DIR is None:
        # Fallback guesswork if traversal fails completely
        BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

print(f"📂 Project Root Absolute Path: {BASE_DIR}")

# Inject Depth-Anything-V2 repository path directly into Python's search path
REPO_PATH = os.path.join(BASE_DIR, 'src', 'models', 'Depth-Anything-V2')
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

# Inject the notebooks directory so Python can locate your .ipynb file
NOTEBOOKS_DIR = os.path.join(BASE_DIR, "notebooks")
if NOTEBOOKS_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOKS_DIR)

# Add parent directory of Depth-Anything-V2 to resolve module naming variants
PARENT_MODELS_DIR = os.path.dirname(REPO_PATH)
if PARENT_MODELS_DIR not in sys.path:
    sys.path.insert(1, PARENT_MODELS_DIR)

try:
    # Reads directly from your DAV2_Hybrid.ipynb notebook file
    from ipynb.fs.full.DAV2_Hybrid import load_hybrid_model
    print("✅ Success: Imported Hybrid Model directly from DAV2_Hybrid.ipynb!")
except ImportError as e:
    print(f"\n[CRITICAL ERROR]: Could not parse notebook features. Error detail: {e}")
    print("👉 Make sure you ran 'pip install ipynb' in your terminal environment.")
    print(f"👉 Verify that 'DAV2_Hybrid.ipynb' is inside: {NOTEBOOKS_DIR}\n")
    if 'get_ipython' not in globals():
        sys.exit(1)

# ==========================================================
# 2. MATCH DATA SET STRUCTURE PATHS
# ==========================================================
TRAIN_DATA_PATH = os.path.join(BASE_DIR, "src", "data", "data", "processed", "train")
VAL_DATA_PATH = os.path.join(BASE_DIR, "src", "data", "data", "processed", "val")

# ==========================================================
# 3. CUSTOM NYU DATASET CLASS WITH ON-THE-FLY PREPROCESSING
# ==========================================================
class NYUDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir = data_dir
        if not os.path.exists(data_dir):
            print(f"⚠️ Warning: Target directory missing -> {data_dir}")
            self.files = []
        else:
            self.files = [f for f in os.listdir(data_dir) if f.endswith('.pt')]
        
        # Preprocessing pipeline required by the DA-V2 ViT Backbone
        self.transform = transforms.Compose([
            transforms.Resize((518, 518)),  # Rigid resolution matching ViT-B expectations
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = os.path.join(self.data_dir, self.files[idx])
        
        # Safe checkpoint loading for PyTorch 2.6+ environments containing NumPy structures
        data = torch.load(path, map_location='cpu', weights_only=False)
        
        # Images were stored as (H, W, C) from NumPy -> Convert to PyTorch (C, H, W)
        image = torch.from_numpy(data['image']).permute(2, 0, 1).float() / 255.0
        image = self.transform(image)
        
        # Depths were stored as (H, W) -> Add explicit channel dimension
        depth = torch.from_numpy(data['depth']).float().unsqueeze(0)
        
        # Interpolate the ground truth target to match the 518x518 processing space
        depth = torch.nn.functional.interpolate(depth.unsqueeze(0), 
                                                size=(518, 518), 
                                                mode='nearest').squeeze(0)
        return image, depth

# ==========================================================
# 4. SCALE-INVARIANT LOGARITHMIC LOSS (SILog)
# ==========================================================
class SILogLoss(torch.nn.Module):
    def __init__(self, lambd=0.5):
        super(SILogLoss, self).__init__()
        self.lambd = lambd

    def forward(self, pred, target):
        valid_mask = (target > 0).detach()
        diff = torch.log(pred[valid_mask] + 1e-6) - torch.log(target[valid_mask] + 1e-6)
        loss = torch.sqrt(torch.mean(diff**2) - self.lambd * (torch.mean(diff)**2))
        return loss

# ==========================================================
# 5. MAIN ENGINE LOOP EXECUTOR
# ==========================================================
def run_retrain_pipeline():
    EPOCHS = 10
    BATCH_SIZE = 4
    LEARNING_RATE = 1e-4

    # Initialize tracking session on Weights & Biases cloud
    print("🔄 Booting experiment tracking metrics system via W&B...")
    wandb.init(
        project="Monocular-3D-Reconstruction", 
        name="hybrid-decoder-retrain-run",
        config={
            "architecture": "Depth-Anything-V2 Hybrid Ensemble",
            "loss_function": "SILogLoss",
            "learning_rate": LEARNING_RATE,
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE
        }
    )
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"--- Console: Training session executing on device target: {device} ---")
    
    # ==========================================================
    # FIXED: LOADING DIRECTLY FROM YOUR SAVED .PTH FILE
    # ==========================================================
    checkpoint_dir = os.path.join(BASE_DIR, "checkpoints")
    checkpoint_path = os.path.join(checkpoint_dir, "latest_hybrid_model.pth")
    
    print(f"🎯 Loading existing hybrid model weights from: {checkpoint_path}")
    
    if not os.path.exists(checkpoint_path):
        print(f"❌ CRITICAL ERROR: Could not find '{checkpoint_path}'.")
        print("👉 Please make sure the 'latest_hybrid_model.pth' file is inside your root checkpoints/ folder.")
        return

    # 1. Initialize the base DepthAnythingV2 structural backbone shell
    from depth_anything_v2.dpt import DepthAnythingV2
    config = {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]}
    
    # 2. Build a clean model class definition holding your custom decoder layout
    class HybridEnsemble(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.backbone = DepthAnythingV2(**config)
            # Match your exact custom decoder layers layout
            self.custom_decoder = torch.nn.Sequential(
                torch.nn.Conv2d(1, 32, kernel_size=3, padding=1),
                torch.nn.ReLU(),
                torch.nn.Conv2d(32, 1, kernel_size=1)
            )
        def forward(self, x):
            with torch.no_grad():
                raw_depth = self.backbone(x)
            if raw_depth.ndim == 3:
                raw_depth = raw_depth.unsqueeze(1)
            return self.custom_decoder(raw_depth)

    # 3. Instantiate the empty shell structure
    model = HybridEnsemble()
    
    # 4. Inject your saved checkpoint weights safely into the structure
    model.load_state_dict(torch.load(checkpoint_path, map_location=device), strict=False)
    print("✅ Success: Existing weights mapped into the Hybrid Ensemble architecture!")

    # Surgical Engineering Strategy: Freeze backbone encoder completely
    print("🔒 Freezing base Transformer encoder backbone weights...")
    for param in model.backbone.parameters():
        param.requires_grad = False

    model = model.to(device)
    
    # Instantiate Data Loaders
    print("📦 Loading local processed dataset components...")
    train_dataset = NYUDataset(TRAIN_DATA_PATH)
    val_dataset = NYUDataset(VAL_DATA_PATH)

    if len(train_dataset) == 0:
        print(f"❌ CRITICAL ERROR: Empty training set. Verification needed at path: {TRAIN_DATA_PATH}")
        print("👉 Did you execute 'dvc pull' to download the processed binaries?")
        return

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    
    # Validation Loader attachment check
    has_validation = len(val_dataset) > 0
    if has_validation:
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
        print(f"✅ Validation partition loaded successfully with {len(val_dataset)} elements.")
    else:
        print("⚠️ Notice: Validation data path is empty or unpopulated. Skipping validation stages.")
    
    # CRITICAL PIPELINE DESIGN: Only optimize parameters bound to the trainable CNN decoder!
    optimizer = optim.Adam(model.custom_decoder.parameters(), lr=LEARNING_RATE)
    criterion = SILogLoss()
    
    # Set up checkpoints output directory path string
    checkpoint_dir = os.path.join(BASE_DIR, "checkpoints")
    os.makedirs(checkpoint_dir, exist_ok=True)
    checkpoint_path = os.path.join(checkpoint_dir, "latest_hybrid_model.pth")

    best_val_loss = float('inf')
    
    print(f"--- Console: Kicking off optimization over {EPOCHS} Epochs ---")
    for epoch in range(EPOCHS):
        # --- Training Loop Execution Stage ---
        model.train()
        epoch_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        
        for batch_idx, (images, depths) in enumerate(pbar):
            images, depths = images.to(device), depths.to(device)
            
            optimizer.zero_grad()
            output = model(images)
            
            # Enforce spatial clamp boundaries to stop logarithmic training divergence
            output = torch.clamp(output, min=0.1, max=10.0)
            
            loss = criterion(output, depths)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
            # Record individual batch steps inside W&B tracking charts
            wandb.log({"batch_loss": loss.item()})
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        avg_epoch_loss = epoch_loss / len(train_loader)
        print(f"--- Epoch {epoch+1} Completed | Avg Training SILog Loss: {avg_epoch_loss:.4f} ---")
        wandb.log({"epoch_loss": avg_epoch_loss, "epoch": epoch + 1})
        
        # --- Validation Stage Evaluation Check ---
        if has_validation:
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for val_images, val_depths in val_loader:
                    val_images, val_depths = val_images.to(device), val_depths.to(device)
                    val_output = model(val_images)
                    val_output = torch.clamp(val_output, min=0.1, max=10.0)
                    v_loss = criterion(val_output, val_depths)
                    val_loss += v_loss.item()
            
            avg_val_loss = val_loss / len(val_loader)
            print(f"📊 Validation Run Metrics -> Avg Loss: {avg_val_loss:.4f}")
            wandb.log({"val_loss": avg_val_loss, "epoch": epoch + 1})
            
            # Checkpointing Strategy: Keep the best generalizing validation loss model
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                torch.save(model.state_dict(), checkpoint_path)
                print(f"💾 Guarded Checkpoint Updated! Saved Best Weights to: {checkpoint_path}")
        else:
            # Fallback local dump tracking strategy if validation data is not tracked
            torch.save(model.state_dict(), checkpoint_path)
            print(f"Checkpoint state dumped locally to: {checkpoint_path}")

    print("\n📦 Registering final optimized weights to W&B cloud storage artifact registry...")
    artifact = wandb.Artifact(
        name="depth_anything_hybrid_ensemble", 
        type="model", 
        description="Metric custom CNN decoder fine-tuned via processed PyTorch binary structures."
    )
    artifact.add_file(checkpoint_path)
    wandb.log_artifact(artifact)

    wandb.finish()
    print("🎉 Retraining Pipeline Workflow Completed Successfully!")

if __name__ == "__main__":
    run_retrain_pipeline()